In [1]:
import json
import os
from tqdm import tqdm

In [2]:
MRTMD_JSON_FILE = "./MRTMD-main/ground_truth/groundtruth_1080p.json"
MRTMD_LABELS_PATH = "./MRTMD-main/labels/1080p"

SEU_PML_INIT_LABEL_PATH = "./SEU_PML/train/labels_init"
SEU_PML_LABELS_PATH = "./SEU_PML/train/labels"

## MRTMD 数据集预处理


In [3]:
def class_map(class_id):
    """
    原始映射: 1 -> person, 2 -> bicycle, 3 -> car, 4 -> motorcycle, 6 -> bus, 8 -> truck
    新映射: 0 -> Person(1), 1 -> Motor Vehicle(3, 6, 8), 2 -> Non-Motor Vehicle(2, 4)
    """
    if class_id == 1:
        return 0
    elif class_id in [3, 6, 8]:
        return 1
    elif class_id in [2, 4]:
        return 2
    else:
        print(f"Warning: Unknown class ID {class_id}")
        return -1

def coco_to_yolo(class_id, img_width, img_height, bbox):
    """
    COCO 格式 (x_min, y_min, width, height) 像素值 -> YOLO 格式
    返回: [class_id, x_center, y_center, width, height] 全部归一化并保留6位小数
    """
    x, y, w, h = bbox
    
    x_center = (x + w / 2.0) / img_width
    y_center = (y + h / 2.0) / img_height
    width = w / img_width
    height = h / img_height
    
    eps = 1e-6
    x_center = max(eps, min(1.0 - eps, x_center))
    y_center = max(eps, min(1.0 - eps, y_center))
    width = max(eps, min(1.0 - eps, width))
    height = max(eps, min(1.0 - eps, height))
    
    return class_id, round(x_center, 6), round(y_center, 6), round(width, 6), round(height, 6)


def process_MRTMD(json_file, labels_path):
    os.makedirs(labels_path, exist_ok=True)

    with open(json_file, "r", encoding="utf-8") as f:
        data_set = json.load(f)
        print(f"数据集键: {data_set.keys()}")

        # 按图片ID分组标注
        annotations_by_image = {}
        for anno in data_set["annotations"]:
            img_id = anno["image_id"]
            if img_id not in annotations_by_image:
                annotations_by_image[img_id] = []
            annotations_by_image[img_id].append(anno)
        print(f"共 {len(annotations_by_image)} 张图片包含标注")

        # 为整个数据集生成YOLO格式的标注文件
        for img in tqdm(data_set["images"], desc="正在转换 YOLO 标签"):
            img_id = img["id"]
            img_width = img["width"]
            img_height = img["height"]
            img_name = img["file_name"]

            annos = annotations_by_image.get(img_id, None)  # 获取该图片对应的所有标注
            if annos is None:
                continue  # 如果该图片没有标注，跳过

            yolo_lines = []
            for anno in annos:
                class_id_raw = anno["category_id"]
                bbox = anno["bbox"]  # [x_min, y_min, width, height]
                
                # 映射类别ID
                class_id_mapped = class_map(class_id_raw)
                if class_id_mapped == -1:
                    continue  # 跳过未知类别

                # 转换为YOLO格式
                class_id, x_center, y_center, width, height = coco_to_yolo(class_id_mapped, img_width, img_height, bbox)
                yolo_lines.append(f"{class_id} {x_center} {y_center} {width} {height}\n")

            if not yolo_lines:
                continue

            # 写入YOLO格式的标注文件
            txt_name = os.path.splitext(img_name)[0] + ".txt"
            txt_path = os.path.join(labels_path, txt_name)

            with open(txt_path, "w", encoding="utf-8") as f_txt:
                f_txt.writelines(yolo_lines)

    print(f"✅ 全部转换完成！YOLO 标签保存在: {labels_path}")

## SEU_SEM 数据集预处理

In [4]:
def process_SEU_PML(input_dir, output_dir, remove_class=3):
    """
    过滤YOLO标签文件中指定类别的行
    remove_class: 要删除的类别ID
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    # 遍历输入目录下所有 .txt 文件
    txt_files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]
    if not txt_files:
        print(f"⚠️ 在 {input_dir} 中没有找到任何 .txt 文件，请检查路径。")
        return

    print(f"找到 {len(txt_files)} 个标签文件，开始处理...")

    for filename in tqdm(txt_files, desc="过滤标签"):
        src_path = os.path.join(input_dir, filename)
        dst_path = os.path.join(output_dir, filename)

        filtered_lines = []
        with open(src_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                part = line.split()

                cls_id = int(part[0])
                if cls_id != remove_class:
                    filtered_lines.append(line)

        if filtered_lines:
            with open(dst_path, 'w', encoding='utf-8') as f_out:
                f_out.write('\n'.join(filtered_lines) + '\n')

    print(f"✅ 处理完成！过滤后的标签保存在: {output_dir}")

In [5]:
if __name__ == "__main__":
    print("="*50)
    print("处理MRTMD数据集")
    print("="*50)
    process_MRTMD(MRTMD_JSON_FILE, MRTMD_LABELS_PATH)
    print("="*50)
    print()

    print("="*50)
    print("处理SEU-PML数据集")
    print("="*50)
    process_SEU_PML(SEU_PML_INIT_LABEL_PATH, SEU_PML_LABELS_PATH, remove_class=3)
    print("="*50)
    print()


处理MRTMD数据集
数据集键: dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])
共 3733 张图片包含标注


正在转换 YOLO 标签: 100%|██████████| 3733/3733 [00:01<00:00, 2621.09it/s]


✅ 全部转换完成！YOLO 标签保存在: ./MRTMD-main/labels/1080p

处理SEU-PML数据集
找到 5269 个标签文件，开始处理...


过滤标签: 100%|██████████| 5269/5269 [00:02<00:00, 2165.89it/s]

✅ 处理完成！过滤后的标签保存在: ./SEU_PML/train/labels

